# Rank and Search: Getting the Right Documents

Level 3: RAG and Retrieval Engineering.

**Problems in this notebook**

- Problem 21: Rank Retrieved Documents
- Problem 22: Top-K Search over Embeddings
- Problem 27: Clean a Search Query

**How to use this notebook**

1. Read the problem and say your plan aloud before you type anything.
2. Paste the AI prompt given (or write your own narrow one) to fill the starter. Read the AI's code before you run it.
3. Run the checks. Open the reference only after you have tried.

The ideas behind these problems are in `01_rank_and_search_THEORY.md` in this folder.

In [ ]:
def run_checks(fn, cases):
    """cases: list of (args_tuple, expected). Prints a short report, never raises."""
    passed = 0
    for i, (args, expected) in enumerate(cases, 1):
        try:
            got = fn(*args)
        except NotImplementedError:
            print("Not implemented yet. Fill in the starter first.")
            return False
        except Exception as e:
            print(f"Case {i}: raised {type(e).__name__}: {e}")
            continue
        if got == expected:
            passed += 1
        else:
            print(f"Case {i}: expected {expected!r}, got {got!r}")
    print(f"{passed}/{len(cases)} checks passed")
    return passed == len(cases)

## Problem 21: Rank Retrieved Documents

**The situation**

A RAG assistant searches the knowledge base two ways at once: vector search gives a meaning score and keyword search gives a word-match score. Only the best few documents go into the prompt, so the order matters.

**Your task**

Write `rank_documents(docs, semantic_weight=0.7)`. Mix the two scores with the weights and return the document IDs, best first.

**Example**

`[{"id": "a", "semantic": 0.1, "keyword": 0.1}, {"id": "b", "semantic": 0.9, "keyword": 0.9}]` -> `["b", "a"]`

**Ask before coding**

- Are both scores already on the same 0 to 1 scale?
- If a document has only one of the two scores, does the missing one count as zero?
- When two documents tie, should they keep the order they came in?

**Plan to say aloud**

1. Check the weight is between 0 and 1. The keyword weight is 1 minus it.
2. Skip any document that has no ID.
3. Score each document: semantic times its weight plus keyword times its weight. Missing score means 0.
4. Sort by score, highest first. Python's sort keeps ties in input order.
5. Return just the IDs.

### Prompt for your AI assistant

```text
Write a Python function rank_documents(docs: list[dict], semantic_weight: float = 0.7) -> list[str].
Each doc is a dict with "id", and optional "semantic" and "keyword" scores (floats).
Rules:
- keyword weight is 1 - semantic_weight
- raise ValueError if semantic_weight is not between 0 and 1
- a missing score counts as 0.0
- skip docs with no "id"
- return IDs sorted by combined score, highest first; ties keep input order
Standard library only. Show only the function.
```

**Check the AI's code for**

- Sorts ascending (forgets `reverse=True`), so the worst document comes first.
- Uses `doc["keyword"]` and crashes on a document with only one score.
- Hard-codes 0.3 for the keyword weight instead of `1 - semantic_weight`.
- Adds a fancy normalisation step you did not ask for, which changes the ranking.

In [ ]:
def rank_documents(docs: list[dict], semantic_weight: float = 0.7) -> list[str]:
    """Return doc IDs ranked by a weighted mix of semantic and keyword scores.

    Rules:
    - keyword weight = 1 - semantic_weight; ValueError if weight is outside 0..1
    - a missing score counts as 0.0
    - docs with no "id" are skipped
    - highest score first; ties keep input order
    """
    raise NotImplementedError

In [ ]:
cases = [
    (([{"id": "d1", "semantic": 0.8, "keyword": 0.2}],), ["d1"]),
    (([{"id": "a", "semantic": 0.1, "keyword": 0.1},
       {"id": "b", "semantic": 0.9, "keyword": 0.9}],), ["b", "a"]),
    # missing scores count as zero: a = 0.35, b = 0.27
    (([{"id": "a", "semantic": 0.5}, {"id": "b", "keyword": 0.9}],), ["a", "b"]),
    # tie keeps input order
    (([{"id": "x", "semantic": 0.5, "keyword": 0.5},
       {"id": "y", "semantic": 0.5, "keyword": 0.5}],), ["x", "y"]),
    # doc with no id is skipped
    (([{"semantic": 0.9}, {"id": "a", "semantic": 0.1}],), ["a"]),
    # weight 0.0 means keyword score only
    (([{"id": "a", "semantic": 1.0, "keyword": 0.0},
       {"id": "b", "semantic": 0.0, "keyword": 1.0}], 0.0), ["b", "a"]),
]
run_checks(rank_documents, cases)

### Reference solution

Try first. Then compare.

In [ ]:
def rank_documents_reference(docs: list[dict], semantic_weight: float = 0.7) -> list[str]:
    if not 0.0 <= semantic_weight <= 1.0:
        raise ValueError("semantic_weight must be between 0 and 1")
    keyword_weight = 1.0 - semantic_weight

    def score(doc: dict) -> float:
        return doc.get("semantic", 0.0) * semantic_weight + doc.get("keyword", 0.0) * keyword_weight

    with_ids = [doc for doc in docs if doc.get("id")]
    return [doc["id"] for doc in sorted(with_ids, key=score, reverse=True)]


assert run_checks(rank_documents_reference, cases)

**Explain it in 30 seconds**

> "I give each document one combined score: the meaning score times its weight plus the keyword score times the rest of the weight. A missing score counts as zero, and documents with no ID are skipped. Then I sort highest first, and ties stay in the order they arrived. Before trusting the mix, I would ask whether the two scores are on the same scale."

**Edge cases to mention**

- Scores on different scales, for example keyword scores from 0 to 30 next to cosine scores from 0 to 1.
- The same document ID returned by both searches as two rows.
- A weight outside 0 to 1, which would flip the meaning of a score.
- Documents the user is not allowed to see. These must be removed before ranking, not after.

**If they ask "how would this work in production?"**

- Normalise or calibrate each score source first, or use rank-based fusion that ignores raw scale.
- Add a reranker model on the top results and track ranking quality on a fixed set of test questions.
- Apply permission filters before ranking so hidden documents never take a top slot.

## Problem 22: Top-K Search over Embeddings

**The situation**

You are building a quick RAG prototype for a customer and do not want to set up a vector database yet. You have a few hundred documents with embedding vectors in memory.

**Your task**

Write `top_k_search(query, docs, k)` that returns the IDs of the `k` documents whose vectors are closest in direction to the query vector (cosine similarity).

**Example**

`query=[1, 0]`, `docs={"d1": [1, 0], "d2": [0, 1]}`, `k=1` -> `["d1"]`

**Ask before coding**

- Should a vector with a different length be an error, or skipped?
- What score should an all-zero vector get?
- Is this only for a prototype size, or will it need to scale?

**Plan to say aloud**

1. Write a small cosine helper: dot product divided by the two lengths.
2. If the lengths of the vectors differ, raise an error. If either vector is all zeros, return 0.
3. Score every document against the query.
4. Sort highest first and keep the first k. If k is 0 or less, return an empty list.

### Prompt for your AI assistant

```text
Write a Python function top_k_search(query: list[float], docs: dict[str, list[float]], k: int) -> list[str].
Rules:
- score each doc by cosine similarity with the query
- raise ValueError if a doc vector has a different length than the query
- if either vector is all zeros, its score is 0.0 (no division by zero)
- return the IDs of the top k docs, highest score first; ties keep dict order
- k <= 0 returns []; k larger than the number of docs returns all of them
Standard library only (math is fine), no numpy. Show only the function and its helper.
```

**Check the AI's code for**

- Divides by the vector length without checking for zero, so an empty embedding crashes the search.
- Uses `zip` without a length check, which silently ignores extra numbers in a longer vector.
- Reaches for numpy or a vector database client when you asked for standard library only.
- Slices with `[:k]` when k is negative, which drops items from the end instead of returning nothing.

In [ ]:
def top_k_search(query: list[float], docs: dict[str, list[float]], k: int) -> list[str]:
    """Return the IDs of the k docs most similar to the query (cosine similarity).

    Rules:
    - ValueError if a doc vector length differs from the query length
    - an all-zero vector scores 0.0
    - highest score first; ties keep dict order
    - k <= 0 returns []
    """
    raise NotImplementedError

In [ ]:
cases = [
    (([1, 0], {"d1": [1, 0], "d2": [0, 1]}, 1), ["d1"]),
    # a and c tie; the tie keeps dict order
    (([1, 1], {"a": [1, 0], "b": [1, 1], "c": [0, 1]}, 2), ["b", "a"]),
    # k bigger than the number of docs
    (([1, 0], {"a": [1, 0]}, 5), ["a"]),
    (([1, 0], {"a": [1, 0]}, 0), []),
    # zero vector scores 0 and does not crash
    (([1, 0], {"z": [0, 0], "a": [0.5, 0]}, 2), ["a", "z"]),
    (([1, 0], {}, 3), []),
]
run_checks(top_k_search, cases)

### Reference solution

Try first. Then compare.

In [ ]:
import math


def cosine(a: list[float], b: list[float]) -> float:
    if len(a) != len(b):
        raise ValueError("vectors must have the same length")
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return sum(x * y for x, y in zip(a, b)) / (norm_a * norm_b)


def top_k_search_reference(query: list[float], docs: dict[str, list[float]], k: int) -> list[str]:
    if k <= 0:
        return []
    ranked = sorted(docs, key=lambda doc_id: cosine(query, docs[doc_id]), reverse=True)
    return ranked[:k]


assert run_checks(top_k_search_reference, cases)

# A length mismatch must be loud, not silently cut short.
try:
    top_k_search_reference([1], {"a": [1, 2]}, 1)
    raise AssertionError("expected ValueError")
except ValueError:
    print("length mismatch rejected")

**Explain it in 30 seconds**

> "Cosine similarity asks whether two vectors point the same way. I compute it for every document, guard against all-zero vectors so I never divide by zero, and reject vectors of the wrong length because that means two different embedding models got mixed. Then I sort and keep the top k. This is fine for a prototype; at scale I would move to a vector index."

**Edge cases to mention**

- A vector of a different length, which usually means a different embedding model was used.
- An all-zero vector, for example from an empty document.
- k of zero, negative k, or k bigger than the number of documents.
- An empty document store.

**If they ask "how would this work in production?"**

- Move to a vector index (pgvector, FAISS, a managed vector database) that finds near neighbours without scanning everything.
- Filter by tenant and permissions inside the search, not after it.
- Store the embedding model name with each vector so a model change cannot mix incompatible vectors.

## Problem 27: Clean a Search Query

**The situation**

Users type messy questions into the assistant: extra spaces, question marks, capital letters. Light cleaning helps keyword search, but some queries contain IDs like `E-404` or `v2.1` that must survive.

**Your task**

Write `preprocess_query(query)` that lowercases, trims, collapses spaces, and strips punctuation from the start and end of each word. Punctuation inside a word stays.

**Example**

`"  What is the Refund Policy???  "` -> `"what is the refund policy"`; `"error E-404 in v2.1?"` -> `"error e-404 in v2.1"`

**Ask before coding**

- Do queries contain product codes, error codes, or version numbers we must keep?
- Should we remove common words like "the"? (Usually no, without testing first.)
- Do we keep the original query for the logs?

**Plan to say aloud**

1. Lowercase the query.
2. Split on any whitespace. This also trims and collapses spaces.
3. Strip punctuation from both ends of each word only.
4. Drop words that became empty, then join with single spaces.

### Prompt for your AI assistant

```text
Write a Python function preprocess_query(query: str) -> str.
Rules:
- lowercase the text
- split on any whitespace (tabs and newlines too) and join with single spaces
- strip punctuation only from the start and end of each word (use string.punctuation)
- keep punctuation inside a word, so "E-404" becomes "e-404" and "v2.1" stays "v2.1"
- drop words that become empty
Standard library only. Show only the function.
```

**Check the AI's code for**

- Removes all punctuation with `str.translate`, which turns `E-404` into `e404` and `v2.1` into `v21`.
- Splits on `" "` instead of plain `split()`, which leaves tabs, newlines, and empty words behind.
- Adds stopword removal or stemming that nobody asked for and nobody has measured.
- Leaves a stray space when a word is only punctuation, like the `???` in `policy ???`.

In [ ]:
def preprocess_query(query: str) -> str:
    """Lightly normalise a search query.

    Rules:
    - lowercase; split on any whitespace; join with single spaces
    - strip punctuation from the start and end of each word only
    - keep punctuation inside a word ("E-404" -> "e-404", "v2.1" stays)
    - drop words that become empty
    """
    raise NotImplementedError

In [ ]:
cases = [
    (("  What is the Refund Policy???  ",), "what is the refund policy"),
    (("error E-404 in v2.1?",), "error e-404 in v2.1"),
    (("reset\tmy   password\n",), "reset my password"),
    (("(see) 'SSO' setup",), "see sso setup"),
    (("?!...",), ""),
    (("",), ""),
]
run_checks(preprocess_query, cases)

### Reference solution

Try first. Then compare.

In [ ]:
import string


def preprocess_query_reference(query: str) -> str:
    words = (word.strip(string.punctuation) for word in query.lower().split())
    return " ".join(word for word in words if word)


assert run_checks(preprocess_query_reference, cases)

**Explain it in 30 seconds**

> "I keep the cleaning light on purpose. I lowercase, split on whitespace so extra spaces vanish, and strip punctuation only from the edges of each word. That way question marks go away but codes like E-404 and v2.1 survive. I would only make it stronger if retrieval tests showed it helps."

**Edge cases to mention**

- A query that is only punctuation, which becomes an empty string. The caller must handle that.
- Codes and IDs with inner punctuation, such as `INV-2024-001` or `user@corp.com`.
- Code snippets, where symbols like `()` carry meaning.
- Other languages, where lowercasing and punctuation rules differ.

**If they ask "how would this work in production?"**

- Keep the original query in the logs next to the cleaned one, so you can debug bad answers.
- Use the search engine's own text analyser per language instead of custom rules.
- Measure retrieval quality before and after any change to the cleaning steps.

## Wrap-up

- All three shape what the retriever finds: the query going in, the similarity score, and the final order.
- Each one has a quiet failure (wrong sort order, zero vectors, broken IDs) that looks fine on the happy path.
- Keep the first version simple and explainable, then improve it only when retrieval tests prove it helps.